In [1]:
import os, time, re, subprocess, warnings
import numpy as np, pandas as pd, torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
warnings.filterwarnings('ignore')
OUT_DIR = '/kaggle/working'
dev = 0 if torch.cuda.is_available() else -1

In [2]:
for r, d, fs in os.walk('/kaggle/input'):
    for f in fs:
        if f == 'config.json' or f.lower().endswith(('.mp4', '.mkv', '.avi', '.mov', '.webm', '.wav', '.mp3')):
            print(os.path.join(r, f))

/kaggle/input/datasets/pavithra1mj/check-latency/WhatsApp Video 2026-10-03 at 11.15.52 AM.mp4
/kaggle/input/notebooks/pavithra1mj/bert-model-by-dataset1/bert_finetuned/config.json
/kaggle/input/notebooks/pavithra1mj/bert-model-by-dataset1/bert_small/config.json


In [7]:
MODEL_DIR = '/kaggle/input/notebooks/pavithra1mj/bert-model-by-dataset1'
VIDEO_PATH = '/kaggle/input/datasets/pavithra1mj/check-latency'

In [10]:
MODEL_DIR = '/kaggle/input/notebooks/pavithra1mj/bert-model-by-dataset1/bert_finetuned'
print(os.listdir(MODEL_DIR))

['config.json', 'training_args.bin', 'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']


In [11]:
t = time.time()
cls_tok = AutoTokenizer.from_pretrained(MODEL_DIR)
cls_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to('cuda' if dev == 0 else 'cpu').eval()
clf = pipeline('text-classification', model=cls_model, tokenizer=cls_tok, device=dev, top_k=None)
asr = pipeline('automatic-speech-recognition', model='openai/whisper-base', device=dev, chunk_length_s=30, return_timestamps=True)
load_time = time.time() - t
print(load_time)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  290MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


11.806530475616455


In [14]:
VIDEO_PATH = '/kaggle/input/datasets/pavithra1mj/check-latency'
print(os.path.exists(VIDEO_PATH))

True


In [20]:
import re
def run(video):
    T = {}
    t0 = time.time()
    wav = f'{OUT_DIR}/audio.wav'
    subprocess.run(['ffmpeg', '-y', '-i', video, '-vn', '-ac', '1', '-ar', '16000', wav], capture_output=True, check=True)
    T['audio_extraction_sec'] = time.time() - t0
    t1 = time.time()
    text = asr(wav)['text'].strip()
    T['transcription_sec'] = time.time() - t1
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    t2 = time.time()
    preds = []
    for s in sents:
        r = max(clf(s)[0], key=lambda x: x['score'])
        preds.append({'sentence': s, 'label': r['label'], 'score': round(r['score'], 4)})
    T['classification_sec'] = time.time() - t2
    T['per_sentence_latency_ms'] = 1000 * T['classification_sec'] / max(1, len(sents))
    T['total_latency_sec'] = time.time() - t0
    return pd.DataFrame(preds), T

In [21]:
print(VIDEO_PATH, os.path.exists(VIDEO_PATH))
r = subprocess.run(['ffmpeg', '-y', '-i', VIDEO_PATH, '-vn', '-ac', '1', '-ar', '16000', f'{OUT_DIR}/audio.wav'], capture_output=True, text=True)
print(r.returncode, r.stderr[-500:])
print(os.path.exists(f'{OUT_DIR}/audio.wav'), os.path.getsize(f'{OUT_DIR}/audio.wav') if os.path.exists(f'{OUT_DIR}/audio.wav') else 0)
out = asr(f'{OUT_DIR}/audio.wav')
print(repr(out['text'][:500]))

/kaggle/input/datasets/pavithra1mj/check-latency True
235 avutil      58. 29.100 / 58. 29.100
  libavcodec     60. 31.102 / 60. 31.102
  libavformat    60. 16.100 / 60. 16.100
  libavdevice    60.  3.100 / 60.  3.100
  libavfilter     9. 12.100 /  9. 12.100
  libswscale      7.  5.100 /  7.  5.100
  libswresample   4. 12.100 /  4. 12.100
  libpostproc    57.  3.100 / 57.  3.100
[in#0 @ 0x55f7017cd700] Error opening input: Is a directory
Error opening input file /kaggle/input/datasets/pavithra1mj/check-latency.
Error opening input files: Is a directory

False 0


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/audio.wav'

In [22]:
print(repr(VIDEO_PATH), os.path.isfile(VIDEO_PATH), os.path.isdir(VIDEO_PATH))
if os.path.isfile(VIDEO_PATH): print(os.path.getsize(VIDEO_PATH) / 1e6, 'MB')
r = subprocess.run(['ffmpeg', '-y', '-i', VIDEO_PATH, '-vn', '-ac', '1', '-ar', '16000', f'{OUT_DIR}/audio.wav'], capture_output=True, text=True)
print(r.returncode)
print(r.stderr[-800:])

'/kaggle/input/datasets/pavithra1mj/check-latency' False True
235
k --enable-libpulse --enable-librabbitmq --enable-librist --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libx264 --enable-libzmq --enable-libzvbi --enable-lv2 --enable-sdl2 --enable-libplacebo --enable-librav1e --enable-pocketsphinx --enable-librsvg --enable-libjxl --enable-shared
  libavutil      58. 29.100 / 58. 29.100
  libavcodec     60. 31.102 / 60. 31.102
  libavformat    60. 16.100 / 60. 16.100
  libavdevice    60.  3.100 / 60.  3.100
  libavfilter     9. 12.100 /  9. 12.100
  libswscale      7.  5.100 /  7.  5.100
  libswresample   4. 12.100 /  4. 12.100
  libpostproc    57.  3.100 / 57.  3.100
[in#0 @ 0x55fa7e9de700] Error opening input: Is a directory
Error opening input file /kaggle/input/datasets/pavithra1mj/check-latency.
Error opening input files: Is a directory



In [23]:
D = '/kaggle/input/datasets/pavithra1mj/check-latency'
files = [os.path.join(r, f) for r, _, fs in os.walk(D) for f in fs]
print(files)
vids = [f for f in files if f.lower().endswith(('.mp4', '.mkv', '.avi', '.mov', '.webm', '.wav', '.mp3', '.m4a', '.flv', '.wmv'))]
VIDEO_PATH = vids[0] if vids else files[0]
print(VIDEO_PATH, os.path.isfile(VIDEO_PATH), os.path.getsize(VIDEO_PATH) / 1e6, 'MB')

['/kaggle/input/datasets/pavithra1mj/check-latency/WhatsApp Video 2026-10-03 at 11.15.52 AM.mp4']
/kaggle/input/datasets/pavithra1mj/check-latency/WhatsApp Video 2026-10-03 at 11.15.52 AM.mp4 True 17.471401 MB


In [24]:
preds_df, T = run(VIDEO_PATH)
print(preds_df)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

                                             sentence   label   score
0   Good morning to all the respected teachers and...  NORMAL  0.9999
1   I'm happy to be a part of this retirement cere...  NORMAL  0.9999
2   I'm truly grateful and lucky to be a student M...  NORMAL  0.9999
3                   Your way of teaching was so good.  NORMAL  0.9999
4   You always gave clear and detailed explanation...  NORMAL  0.9999
5   And one because of you I was able to score a c...  NORMAL  0.9997
6   After attending the event too, she would appre...  NORMAL  0.9998
7   So thank you for being a wonderful teacher and...  NORMAL  0.9999
8           We will always cherish your guidance now.  NORMAL  0.9999
9   Once again, wishing you a very happy, healthy ...  NORMAL  0.9999
10                                         Thank you.  NORMAL  0.9999


In [25]:
preds_df, T = run(VIDEO_PATH)
dur = float(subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', VIDEO_PATH], capture_output=True, text=True).stdout.strip())
T['video_duration_sec'] = dur
T['real_time_factor'] = T['total_latency_sec'] / dur
T['num_sentences'] = len(preds_df)
T['model_load_sec'] = load_time
lat = pd.DataFrame([{k: round(v, 3) for k, v in T.items()}])
lat.to_csv(f'{OUT_DIR}/latency.csv', index=False)
preds_df.to_csv(f'{OUT_DIR}/video_predictions.csv', index=False)
print(lat.T)

                              0
audio_extraction_sec      0.518
transcription_sec         5.019
classification_sec        0.093
per_sentence_latency_ms   8.449
total_latency_sec         5.630
video_duration_sec       78.812
real_time_factor          0.071
num_sentences            11.000
model_load_sec           11.807


In [26]:
import base64
from IPython.display import HTML, display

for f in ['latency.csv', 'video_predictions.csv']:
    p = f'/kaggle/working/{f}'
    b = base64.b64encode(open(p, 'rb').read()).decode()
    display(HTML(f'<a download="{f}" href="data:application/octet-stream;base64,{b}">Download {f}</a>'))